# Practice - 02 - Joins, unions, nested data, and Gold modeling

Prerequisites / execution context: Run 00 and 01. Replaces Gold exercise outputs.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R20: DataFrame joins](https://docs.databricks.com/aws/en/pyspark/reference/classes/dataframe/join); [R21: DataFrame union](https://docs.databricks.com/aws/en/pyspark/reference/classes/dataframe/union); [R22: DataFrame unionByName](https://docs.databricks.com/aws/en/pyspark/reference/classes/dataframe/unionByName); [R35: Materialized views](https://docs.databricks.com/aws/en/ldp/concepts/materialized-views); [R36: Streaming tables](https://docs.databricks.com/aws/en/ldp/concepts/streaming-tables).


**Try first:** complete each YOUR CODE / YOUR ANSWER cell. Only then run the solution-display cell and click Reveal. Solutions are feedback text, never automatically executed. Cells marked check compare your own implementation. Work top to bottom; do not use Run All while placeholders remain.

**Isolation:** the default prefix in this edition is dea_riccardo_practice. Run the practice setup before later practice notebooks. For opt-in cells, complete the code before enabling the flag.


In [0]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "my_catalog"
PREFIX = "dea"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [0]:
# Run once. This helper displays a solution; it never executes solution code.
import base64 as _practice_base64

def show_solution(encoded_html):
    panel = _practice_base64.b64decode(encoded_html).decode("utf-8")
    try:
        renderer = displayHTML  # Databricks
    except NameError:
        from IPython.display import display, HTML
        display(HTML(panel))
    else:
        renderer(panel)


## Coding task 1: Enrich with customer attributes

Load SILVER.orders as o and SILVER.customers as c. Build inner, left and anti joins by customer_id, plus a cross join. Broadcast the dimension in the left join. Predict all row counts before execution; then inspect the effect of filtering country to IT after the left join.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [0]:
# YOUR CODE HERE
o = spark.table(f"{SILVER}.orders")
c = spark.table(f"{SILVER}.customers")

# INNER
spark.sql(f"SELECT * FROM {SILVER}.orders o INNER JOIN {SILVER}.customers c ON o.customer_id = c.customer_id").show()

# LEFT
spark.sql(f"SELECT * FROM {SILVER}.orders o LEFT JOIN {SILVER}.customers c ON o.customer_id = c.customer_id").show()

# CROSS
spark.sql(f"SELECT * FROM {SILVER}.orders o CROSS JOIN {SILVER}.customers c").show()

# ANTI
spark.sql(f"SELECT * FROM {SILVER}.orders o LEFT ANTI JOIN {SILVER}.customers c ON o.customer_id = c.customer_id").show()

# Replace the following line with your implementation.
#raise NotImplementedError("Complete this exercise before continuing")


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDEuIFByZWRpY3Qgam9pbiBjb3VudHMgYmVmb3JlIHJ1bm5pbmcKVXNlIHRoZSBsYXRlc3Qgb3JkZXJzIGFuZCBhIGRpbWVuc2lvbiB3aXRoIHVuaXF1ZSBjdXN0b21lciBJRHMuIEV4cGVjdGVkOiBpbm5lciAyLCBsZWZ0IDQsIGFudGkgMiwgY3Jvc3MgMTIuIEEgbGVmdCBqb2luIHByZXNlcnZlcyB1bm1hdGNoZWQgb3JkZXJzLgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPm89c3BhcmsudGFibGUoZiZxdW90O3tTSUxWRVJ9Lm9yZGVycyZxdW90OykKYz1zcGFyay50YWJsZShmJnF1b3Q7e1NJTFZFUn0uY3VzdG9tZXJzJnF1b3Q7KQppbm5lcj1vLmpvaW4oYywmcXVvdDtjdXN0b21lcl9pZCZxdW90OywmcXVvdDtpbm5lciZxdW90OykKbGVmdD1vLmpvaW4oRi5icm9hZGNhc3QoYyksWyZxdW90O2N1c3RvbWVyX2lkJnF1b3Q7XSwmcXVvdDtsZWZ0JnF1b3Q7KQphbnRpPW8uam9pbihjLCZxdW90O2N1c3RvbWVyX2lkJnF1b3Q7LCZxdW90O2xlZnRfYW50aSZxdW90OykKYXNzZXJ0IChpbm5lci5jb3VudCgpLGxlZnQuY291bnQoKSxhbnRpLmNvdW50KCksby5jcm9zc0pvaW4oYykuY291bnQoKSk9PSgyLDQsMiwxMikKYW50aS5zZWxlY3QoJnF1b3Q7b3JkZXJfaWQmcXVvdDssJnF1b3Q7Y3VzdG9tZXJfaWQmcXVvdDspLnNob3coKQojIEZpbHRlcmluZyBhIHJpZ2h0LXNpZGUgY29sdW1uIGFmdGVyIGEgbGVmdCBqb2luIHJlbW92ZXMgdGhlIHVubWF0Y2hlZCByb3dzLgphc3NlcnQgbGVmdC5maWx0ZXIoJnF1b3Q7Y291bnRyeSA9ICYjeDI3O0lUJiN4Mjc7JnF1b3Q7KS5jb3VudCgpPT0xCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Coding task 2: Diagnose fanout and composite keys

Create dup_c by adding another copy of customer 00035 to c. Measure the resulting left-join count. Build a small fact with (00035,IT,1) and (00035,FR,2), then join to c by both customer_id and country. Explain the difference.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [0]:
# YOUR CODE HERE

spark.sql(f"CREATE OR REPLACE TABLE my_catalog.dea_silver.dup_c AS SELECT * FROM {SILVER}.customers")
spark.sql(f"INSERT INTO {SILVER}.dup_c values ('00035', 'IT')")

spark.sql(f"SELECT * FROM {SILVER}.dup_c").show()

# Replace the following line with your implementation.
#raise NotImplementedError("Complete this exercise before continuing")


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDIuIEV4cG9zZSBkaW1lbnNpb24gZmFub3V0IGFuZCBhIG11bHRpcGxlLWtleSBqb2luClRoaXMgbWlycm9ycyByb3ctY291bnQgcmVjb25jaWxpYXRpb24gZHVyaW5nIG1pZ3JhdGlvbnMuIEEgYnJvYWRjYXN0IGhpbnQgY2hhbmdlcyB0aGUgcGh5c2ljYWwgc3RyYXRlZ3ksIG5vdCB0aGUgaW50ZW5kZWQgcm93cy4gT2ZmaWNpYWwgRGF0YWJyaWNrcyBkb2N1bWVudGF0aW9uOiA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL3B5c3BhcmsvcmVmZXJlbmNlL2NsYXNzZXMvZGF0YWZyYW1lL2pvaW4iIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SMjA6IERhdGFGcmFtZSBqb2luczwvYT4uCgoKVGhlIHJlZmVyZW5jZSBpbXBsZW1lbnRhdGlvbiBpcyBzaG93biBuZXh0LiBDb21wYXJlIHlvdXIgZGVzaWduIGFuZCByZXN1bHRzIGJlZm9yZSBjb3B5aW5nIGl0LjwvZGl2PjwvZGV0YWlscz4=')


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmR1cF9jPWMudW5pb25CeU5hbWUoYy5maWx0ZXIoJnF1b3Q7Y3VzdG9tZXJfaWQ9JiN4Mjc7MDAwMzUmI3gyNzsmcXVvdDspKQphc3NlcnQgby5qb2luKGR1cF9jLCZxdW90O2N1c3RvbWVyX2lkJnF1b3Q7LCZxdW90O2xlZnQmcXVvdDspLmNvdW50KCk9PTUKZmFjdD1zcGFyay5jcmVhdGVEYXRhRnJhbWUoWygmcXVvdDswMDAzNSZxdW90OywmcXVvdDtJVCZxdW90OywxKSwoJnF1b3Q7MDAwMzUmcXVvdDssJnF1b3Q7RlImcXVvdDssMildLCZxdW90O2N1c3RvbWVyX2lkIFNUUklORywgY291bnRyeSBTVFJJTkcsIG4gSU5UJnF1b3Q7KQphc3NlcnQgZmFjdC5qb2luKGMsWyZxdW90O2N1c3RvbWVyX2lkJnF1b3Q7LCZxdW90O2NvdW50cnkmcXVvdDtdLCZxdW90O2lubmVyJnF1b3Q7KS5jb3VudCgpPT0xCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Coding task 3: Union columns correctly

Create a with (id=1,label=x), and b with (label=x,id=1), (label=y,id=2), in that column order. Combine by column name as u, compare counts before/after distinct, and show equivalent SQL UNION and UNION ALL via temp views.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [0]:
# YOUR CODE HERE


# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [0]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert u.count() == 3
assert u.distinct().count() == 2
print("Checks passed")


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDMuIFNRTCBVTklPTiB2ZXJzdXMgRGF0YUZyYW1lIHVuaW9uClByZWRpY3QgdGhyZWUgdmVyc3VzIHR3byByb3dzOyB1c2UgdW5pb25CeU5hbWUgd2hlbiBjb2x1bW5zIGhhdmUgbW92ZWQuIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9weXNwYXJrL3JlZmVyZW5jZS9jbGFzc2VzL2RhdGFmcmFtZS91bmlvbiIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIyMTogRGF0YUZyYW1lIHVuaW9uPC9hPjsgPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9weXNwYXJrL3JlZmVyZW5jZS9jbGFzc2VzL2RhdGFmcmFtZS91bmlvbkJ5TmFtZSIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIyMjogRGF0YUZyYW1lIHVuaW9uQnlOYW1lPC9hPi4KCgpUaGUgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9uIGlzIHNob3duIG5leHQuIENvbXBhcmUgeW91ciBkZXNpZ24gYW5kIHJlc3VsdHMgYmVmb3JlIGNvcHlpbmcgaXQuPC9kaXY+PC9kZXRhaWxzPg==')


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmE9c3BhcmsuY3JlYXRlRGF0YUZyYW1lKFsoMSwmcXVvdDt4JnF1b3Q7KV0sJnF1b3Q7aWQgSU5ULCBsYWJlbCBTVFJJTkcmcXVvdDspCmI9c3BhcmsuY3JlYXRlRGF0YUZyYW1lKFsoJnF1b3Q7eCZxdW90OywxKSwoJnF1b3Q7eSZxdW90OywyKV0sJnF1b3Q7bGFiZWwgU1RSSU5HLCBpZCBJTlQmcXVvdDspCnU9YS51bmlvbkJ5TmFtZShiKQphc3NlcnQgdS5jb3VudCgpPT0zIGFuZCB1LmRpc3RpbmN0KCkuY291bnQoKT09MgphLmNyZWF0ZU9yUmVwbGFjZVRlbXBWaWV3KCZxdW90O3VuaW9uX2EmcXVvdDspOyBiLmNyZWF0ZU9yUmVwbGFjZVRlbXBWaWV3KCZxdW90O3VuaW9uX2ImcXVvdDspCmFzc2VydCBzcGFyay5zcWwoJnF1b3Q7U0VMRUNUIGlkLGxhYmVsIEZST00gdW5pb25fYSBVTklPTiBTRUxFQ1QgaWQsbGFiZWwgRlJPTSB1bmlvbl9iJnF1b3Q7KS5jb3VudCgpPT0yCmFzc2VydCBzcGFyay5zcWwoJnF1b3Q7U0VMRUNUIGlkLGxhYmVsIEZST00gdW5pb25fYSBVTklPTiBBTEwgU0VMRUNUIGlkLGxhYmVsIEZST00gdW5pb25fYiZxdW90OykuY291bnQoKT09Mwo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Coding task 4: Unpack and reshape

Using o, compare row counts after explode versus explode_outer on tags. Split order_id on the letter o into order_parts, rename seq to source_sequence, and drop tags in a separate projection.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [0]:
# YOUR CODE HERE


# Replace the following line with your implementation.
#raise NotImplementedError("Complete this exercise before continuing")


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayA0IGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDQuIEFycmF5cyBhbmQgc3RyaW5ncwpDb21wYXJlIGV4cGxvZGUgd2l0aCBleHBsb2RlX291dGVyIG9uIHRoZSBlbXB0eSB0YWdzIGFycmF5LiBVc2UgYnVpbHQtaW4gZXhwcmVzc2lvbnMgdG8gcmVuYW1lLCBzcGxpdCwgYWRkLCBhbmQgZHJvcCBjb2x1bW5zLiBPZmZpY2lhbCBEYXRhYnJpY2tzIGRvY3VtZW50YXRpb246IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vcHlzcGFyay9iYXNpY3MiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SMTk6IFB5U3BhcmsgYmFzaWNzPC9hPi4KCgpUaGUgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9uIGlzIHNob3duIG5leHQuIENvbXBhcmUgeW91ciBkZXNpZ24gYW5kIHJlc3VsdHMgYmVmb3JlIGNvcHlpbmcgaXQuPC9kaXY+PC9kZXRhaWxzPg==')


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayA0IHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmFzc2VydCBvLnNlbGVjdCgmcXVvdDtvcmRlcl9pZCZxdW90OyxGLmV4cGxvZGUoJnF1b3Q7dGFncyZxdW90OykuYWxpYXMoJnF1b3Q7dGFnJnF1b3Q7KSkuY291bnQoKT09Mwphc3NlcnQgby5zZWxlY3QoJnF1b3Q7b3JkZXJfaWQmcXVvdDssRi5leHBsb2RlX291dGVyKCZxdW90O3RhZ3MmcXVvdDspLmFsaWFzKCZxdW90O3RhZyZxdW90OykpLmNvdW50KCk9PTQKKG8ud2l0aENvbHVtbigmcXVvdDtvcmRlcl9wYXJ0cyZxdW90OyxGLnNwbGl0KCZxdW90O29yZGVyX2lkJnF1b3Q7LCZxdW90O28mcXVvdDspKQogLndpdGhDb2x1bW5SZW5hbWVkKCZxdW90O3NlcSZxdW90OywmcXVvdDtzb3VyY2Vfc2VxdWVuY2UmcXVvdDspLmRyb3AoJnF1b3Q7dGFncyZxdW90Oykuc2hvdygpKQo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Coding task 5: Publish a Gold summary

Using left, map missing countries to UNKNOWN. Build gold at one row per country with orders and revenue. Write GOLD.revenue and create GOLD.revenue_view over it. Calculate the grand total to verify your result.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [0]:
# YOUR CODE HERE
from pyspark.sql import functions as F

silver = spark.table(f"{SILVER}.orders")

gold = (silver.join(spark.table(f"{SILVER}.customers"), "customer_id", 'left')
             .withColumn("country", F.when(F.col("country").isNull(), F.lit("UNKNOWN")).otherwise(F.col("country")))
             .groupBy("country")
             .agg(F.sum("amount").alias("total_revenue")))

gold.show()

# Replace the following line with your implementation.
#raise NotImplementedError("Complete this exercise before continuing")


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayA1IGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDUuIFB1Ymxpc2ggR29sZCBhdCBhIGRlY2xhcmVkIGdyYWluClRoZSBvdXRwdXQgZ3JhaW4gaXMgb25lIGNvdW50cnkgYnVja2V0LiBVbmtub3duIGluY2x1ZGVzIG8zIGFuZCBvNjsgcmV2ZW51ZSBzaG91bGQgYmUgRlIgNTAsIElUIDEyMCwgVU5LTk9XTiA1MCwgdG90YWwgMjIwLiBUaGlzIGxhYiB1c2VzIGEgcmVndWxhciB0YWJsZSBhbmQgdmlldzsgbGFiIDA3IGJ1aWxkcyBhIG1hdGVyaWFsaXplZCB2aWV3LgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayA1IHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmdvbGQ9KGxlZnQud2l0aENvbHVtbigmcXVvdDtjb3VudHJ5JnF1b3Q7LEYuY29hbGVzY2UoJnF1b3Q7Y291bnRyeSZxdW90OyxGLmxpdCgmcXVvdDtVTktOT1dOJnF1b3Q7KSkpCiAuZ3JvdXBCeSgmcXVvdDtjb3VudHJ5JnF1b3Q7KS5hZ2coRi5jb3VudCgmcXVvdDsqJnF1b3Q7KS5hbGlhcygmcXVvdDtvcmRlcnMmcXVvdDspLEYuc3VtKCZxdW90O2Ftb3VudCZxdW90OykuYWxpYXMoJnF1b3Q7cmV2ZW51ZSZxdW90OykpKQpnb2xkLndyaXRlLm1vZGUoJnF1b3Q7b3ZlcndyaXRlJnF1b3Q7KS5zYXZlQXNUYWJsZShmJnF1b3Q7e0dPTER9LnJldmVudWUmcXVvdDspCnNwYXJrLnNxbChmJnF1b3Q7Q1JFQVRFIE9SIFJFUExBQ0UgVklFVyB7R09MRH0ucmV2ZW51ZV92aWV3IEFTIFNFTEVDVCAqIEZST00ge0dPTER9LnJldmVudWUmcXVvdDspCmFzc2VydCBzdHIoZ29sZC5hZ2coRi5zdW0oJnF1b3Q7cmV2ZW51ZSZxdW90OykpLmZpcnN0KClbMF0pPT0mcXVvdDsyMjAuMDAmcXVvdDsKZ29sZC5vcmRlckJ5KCZxdW90O2NvdW50cnkmcXVvdDspLnNob3coKQo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Written / design task 1

Why can exploding tags before aggregating revenue overcount money? State the grain before and after explosion and propose a correct design.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [0]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDEgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgQ2hhbGxlbmdlCkV4cGxhaW4gd2h5IGFnZ3JlZ2F0aW5nIGFmdGVyIGV4cGxvZGluZyB0YWdzIGNhbiBvdmVyY291bnQgcmV2ZW51ZS4gU29sdXRpb246IGV4cGxvc2lvbiBjaGFuZ2VzIHRoZSByb3cgZ3JhaW47IGFnZ3JlZ2F0ZSBhdCB0aGUgb3JpZ2luYWwgb3JkZXIgZ3JhaW4gb3IgYWxsb2NhdGUgdGhlIGFtb3VudCBpbnRlbnRpb25hbGx5Lgo8L2Rpdj48L2RldGFpbHM+')
